In [19]:
!pip install copick

In [20]:
# Make a copick project

config_blob = """{
    "name": "czii_cryoet_mlchallenge_2024",
    "description": "2024 CZII CryoET ML Challenge training data.",
    "data_dir": "/kaggle/input/czii-cryo-et-object-identification/train",
    "num_samples": 100,
    "batch_size": 32,
    "num_classes": 5,
    "learning_rate": 0.0001,
    "epochs": 5,
    "version": "1.0.0",

    "pickable_objects": [
        {
            "name": "apo-ferritin",
            "is_particle": true,
            "pdb_id": "4V1W",
            "label": 1,
            "color": [  0, 117, 220, 128],
            "radius": 60,
            "map_threshold": 0.0418
        },
        {
            "name": "beta-amylase",
            "is_particle": true,
            "pdb_id": "1FA2",
            "label": 2,
            "color": [153,  63,   0, 128],
            "radius": 65,
            "map_threshold": 0.035
        },
        {
            "name": "beta-galactosidase",
            "is_particle": true,
            "pdb_id": "6X1Q",
            "label": 3,
            "color": [ 76,   0,  92, 128],
            "radius": 90,
            "map_threshold": 0.0578
        },
        {
            "name": "ribosome",
            "is_particle": true,
            "pdb_id": "6EK0",
            "label": 4,
            "color": [  0,  92,  49, 128],
            "radius": 150,
            "map_threshold": 0.0374
        },
        {
            "name": "thyroglobulin",
            "is_particle": true,
            "pdb_id": "6SCJ",
            "label": 5,
            "color": [ 43, 206,  72, 128],
            "radius": 130,
            "map_threshold": 0.0278
        },
        {
            "name": "virus-like-particle",
            "is_particle": true,
            "label": 6,
            "color": [255, 204, 153, 128],
            "radius": 135,
            "map_threshold": 0.201
        },
        {
            "name": "membrane",
            "is_particle": false,
            "label": 8,
            "color": [100, 100, 100, 128]
        },
        {
            "name": "background",
            "is_particle": false,
            "label": 9,
            "color": [10, 150, 200, 128]
        }
    ],

    "overlay_root": "/kaggle/working/overlay",

    "overlay_fs_args": {
        "auto_mkdir": true
    },

    "static_root": "/kaggle/input/czii-cryo-et-object-identification/train/static"
}"""

copick_config_path = "/kaggle/working/copick.config"
output_overlay = "/kaggle/working/overlay"

with open(copick_config_path, "w") as f:
    f.write(config_blob)
    


# Import Libraries

In [21]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import zarr
import gc
from torch.cuda.amp import autocast, GradScaler

#    for i, (inputs, labels) in enumerate(train_loader):
       inputs = inputs.to(device)
       labels = labels.to(device).float()
       print("Input shape:", inputs.shape)
       print("Label shape:", labels.shape)
       print("Label example:", labels[0]) # Print one example label
Input Parameters

In [22]:
# Define paths (adjust these if necessary)
train_dir = "/kaggle/input/czii-cryo-et-object-identification/train"  # Path to the train directory
output_model_path = "model.pth" # Path to save the model
tomogram_dir = os.path.join(train_dir, "static", "ExperimentRuns")
overlay_dir = os.path.join(train_dir, "overlay", "ExperimentRuns")
voxel_size = 10

# Set the device (GPU)
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

In [38]:
class CryoETDataset(Dataset):
    def __init__(self, base_dir, patch_size=64):
        """
        Args:
            base_dir (str): Base directory containing both static and overlay dirs
            patch_size (int): Size of the cubic patches to extract
        """
        self.base_dir = base_dir
        self.patch_size = patch_size
        self.tomograms = []
        self.targets = []
        
        # Get all experiment runs
        static_dir = os.path.join(base_dir, 'static', 'ExperimentRuns')
        overlay_dir = os.path.join(base_dir, 'overlay', 'ExperimentRuns')
        
        for exp_name in os.listdir(static_dir):
            exp_path = os.path.join(static_dir, exp_name, 'VoxelSpacing10.000')
            if not os.path.isdir(exp_path):
                continue
                
            # Find denoised.zarr file
            denoised_path = os.path.join(exp_path, 'denoised.zarr')
            if not os.path.exists(denoised_path):
                continue
                
            # Get corresponding picks directory
            picks_dir = os.path.join(overlay_dir, exp_name, 'Picks')
            if not os.path.exists(picks_dir):
                continue
                
            # Get shape information without loading the full tomogram
            zarr_file = zarr.open(denoised_path, mode='r')
            shape = zarr_file['0'].shape
            
            # Calculate number of patches with 50% overlap
            stride = patch_size // 2
            n_patches_x = (shape[0] - patch_size) // stride + 1
            n_patches_y = (shape[1] - patch_size) // stride + 1
            n_patches_z = (shape[2] - patch_size) // stride + 1
            
            # Get all particle type JSON files
            particle_jsons = [f for f in os.listdir(picks_dir) if f.endswith('.json')]
            
            # Store metadata for each patch
            for x in range(n_patches_x):
                for y in range(n_patches_y):
                    for z in range(n_patches_z):
                        patch_info = {
                            'tomogram_path': denoised_path,
                            'position': (x * stride, y * stride, z * stride),
                            'exp_name': exp_name
                        }
                        self.tomograms.append(patch_info)
                        
                        # Store corresponding particle JSONs
                        patch_targets = []
                        for particle_json in particle_jsons:
                            json_path = os.path.join(picks_dir, particle_json)
                            patch_targets.append(json_path)
                        self.targets.append(patch_targets)
            
            print(f"Processed experiment {exp_name}: {len(particle_jsons)} particle types")

    def __len__(self):
        return len(self.tomograms)

    def __getitem__(self, idx):
        # Load patch of data
        tomogram_info = self.tomograms[idx]
        tomogram = zarr.open(tomogram_info['tomogram_path'], mode='r')['0']
        
        x, y, z = tomogram_info['position']
        patch = tomogram[x:x+self.patch_size, 
                        y:y+self.patch_size, 
                        z:z+self.patch_size]
        
        # Convert to tensor and normalize
        patch = torch.from_numpy(patch).float()
        patch = (patch - patch.mean()) / (patch.std() + 1e-6)
        patch = patch.unsqueeze(0)  # Add channel dimension
        
        # Initialize label tensor (5 classes - excluding beta-amylase)
        label = torch.zeros(5, dtype=torch.long)
        
        # Process each particle JSON
        patch_center = np.array([x + self.patch_size//2, 
                               y + self.patch_size//2, 
                               z + self.patch_size//2])
        
        for json_path in self.targets[idx]:
            try:
                with open(json_path, 'r') as f:
                    particle_data = json.load(f)
                    
                # Skip beta-amylase (label 2) as it's not scored
                if 'label' in particle_data and particle_data['label'] != 2:
                    # Adjust label index (skip label 2 in indexing)
                    label_idx = particle_data['label'] - 1
                    if label_idx >= 2:  # After beta-amylase
                        label_idx -= 1
                        
                    # Check if any particle coordinate is within patch
                    if 'coordinates' in particle_data:
                        coords = np.array(particle_data['coordinates'])
                        distances = np.linalg.norm(coords - patch_center, axis=1)
                        if np.any(distances < self.patch_size//2):
                            label[label_idx] = 1
            except Exception as e:
                print(f"Error processing {json_path}: {str(e)}")
                continue
        
        return patch, label

In [54]:
class LightResNet3D(nn.Module):
    def __init__(self, in_channels=1, num_classes=5):  # 5 classes (excluding beta-amylase)
        super(LightResNet3D, self).__init__()
        
        self.encoder = nn.Sequential(
            nn.Conv3d(in_channels, 32, kernel_size=3, padding=1),
            nn.BatchNorm3d(32),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),
            
            nn.Conv3d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm3d(64),
            nn.ReLU(inplace=True),
            nn.MaxPool3d(2),
            
            nn.Conv3d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm3d(128),
            nn.ReLU(inplace=True),
            nn.AdaptiveAvgPool3d((1, 1, 1))
        )
        
        self.classifier = nn.Linear(128, num_classes)

    def forward(self, x):
        x = self.encoder(x)
        x = x.view(x.size(0), -1)
        x = self.classifier(x)
        return x


In [57]:
def train_model(model, train_loader, num_epochs=5, device='cuda'):
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)
    criterion = nn.BCEWithLogitsLoss()  # Better for multi-label classification
    scaler = GradScaler()
    
    for epoch in range(num_epochs):
        model.train()
        running_loss = 0.0
        
        for i, (inputs, labels) in enumerate(train_loader):
            inputs = inputs.to(device)
            labels = labels.to(device).float()
            # print("Input shape:", inputs.shape)
            # print("Label shape:", labels.shape)
            # print("Label example:", labels[0]) # Print one example label
            
            optimizer.zero_grad()
            
            # Mixed precision training
            with autocast():
                outputs = model(inputs)
                loss = criterion(outputs, labels)
            
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
            
            running_loss += loss.item()
            
            if i % 10 == 9:
                print(f'[{epoch + 1}, {i + 1}] loss: {running_loss / 10:.3f}')
                running_loss = 0.0
            
            # Clear cache periodically
            if i % 50 == 0:
                torch.cuda.empty_cache()
                gc.collect()

In [ ]:
# Parameters
base_dir = '/kaggle/input/czii-cryo-et-object-identification/train'
patch_size = 64
batch_size = 4
num_epochs = 5

# Initialize dataset
dataset = CryoETDataset(base_dir=base_dir, patch_size=patch_size)

# Create dataloader
train_loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# Initialize model
model = LightResNet3D()

# Set memory efficient options
torch.backends.cudnn.benchmark = True
torch.cuda.empty_cache()

# Train model
train_model(model, train_loader, num_epochs=num_epochs)

Processed experiment TS_86_3: 6 particle types
Processed experiment TS_6_6: 6 particle types
Processed experiment TS_6_4: 6 particle types
Processed experiment TS_5_4: 6 particle types
Processed experiment TS_73_6: 6 particle types
Processed experiment TS_99_9: 6 particle types
Processed experiment TS_69_2: 6 particle types


/tmp/ipykernel_30/3099613442.py:5: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_30/3099613442.py:21: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[1, 10] loss: 0.643
[1, 20] loss: 0.602
[1, 30] loss: 0.566
[1, 40] loss: 0.538
[1, 50] loss: 0.505
[1, 60] loss: 0.482
[1, 70] loss: 0.459
[1, 80] loss: 0.438
[1, 90] loss: 0.419
[1, 100] loss: 0.397
[1, 110] loss: 0.381
[1, 120] loss: 0.364
[1, 130] loss: 0.350
[1, 140] loss: 0.340
[1, 150] loss: 0.325
[1, 160] loss: 0.313
[1, 170] loss: 0.299
[1, 180] loss: 0.290
[1, 190] loss: 0.278
[1, 200] loss: 0.268
[1, 210] loss: 0.258
[1, 220] loss: 0.250
[1, 230] loss: 0.241
[1, 240] loss: 0.232
[1, 250] loss: 0.223
[1, 260] loss: 0.215
[1, 270] loss: 0.209
[1, 280] loss: 0.199
[1, 290] loss: 0.192
[1, 300] loss: 0.187
[1, 310] loss: 0.181
[1, 320] loss: 0.173
[1, 330] loss: 0.169
[1, 340] loss: 0.162
[1, 350] loss: 0.157
[1, 360] loss: 0.152
[1, 370] loss: 0.147
[1, 380] loss: 0.142
[1, 390] loss: 0.138
[1, 400] loss: 0.133
[1, 410] loss: 0.129
[1, 420] loss: 0.125
[1, 430] loss: 0.121
[1, 440] loss: 0.117
[1, 450] loss: 0.114
[1, 460] loss: 0.111
[1, 470] loss: 0.108
[1, 480] loss: 0.104
[